In [ ]:


import pandas as pd
from pathlib import Path
from datetime import date
import yaml

PATH_SERVER = Path(r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring\Data")
PATH_SP = Path.home()/r'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents\Data\Vibrant and Inclusive Places\Economy\Income\Income_3 By Race'
PATH_CONFIG0 = Path.home()/r'Documents\Projects\Regional-Monitoring\Indicator_Gen\config'


EXPORT=False
ABOUT=False
UPDATE=False


EST='ACS1'
GEO='Counties'



def clean_fips(df):
        
    '''
    FIPS codes are often used across data sources but they don't always come in the same format, especially when using different file types (.csv, .xlsx, ...)
    This function standardizes the FIPS format for various FIPS codes
    '''

    print('Cleaning FIPS codes to standard format...')
    
    if 'STATEFP' in df.columns:                          df['STATEFP'                         ] = df['STATEFP'                         ].astype(str).apply('{:0>2}'.format)
    if 'State FIPS' in df.columns:                       df['State FIPS'                      ] = df['State FIPS'                      ].astype(str).apply('{:0>2}'.format)
    if 'Place ID' in df.columns:                         df['Place ID'                        ] = df['Place ID'                        ].astype(str).apply('{:0>5}'.format)
    if 'COUNTYFP' in df.columns:                         df['COUNTYFP'                        ] = df['COUNTYFP'                        ].astype(str).apply('{:0>3}'.format)
    if 'County FIPS' in df.columns:                      df['County FIPS'                     ] = df['County FIPS'                     ].astype(str).apply('{:0>3}'.format)
    if 'Congressional District' in df.columns:           df['Congressional District'          ] = df['Congressional District'          ].astype(str).apply('{:0>2}'.format)
    if 'State Legislative Upper District' in df.columns: df['State Legislative Upper District'] = df['State Legislative Upper District'].astype(str).apply('{:0>3}'.format)
    if 'State Legislative Lower District' in df.columns: df['State Legislative Lower District'] = df['State Legislative Lower District'].astype(str).apply('{:0>3}'.format)

    return df




# Writes about page for each params['indicator']
def write_about(df_inc3):

    '''
    User defined function to create/export about documentation for each params['indicator']
    Inputs: yaml file, specific params['indicator'] inputs (params['geo'], sample type, ...), data frame to export, file paths, ...
    Uses user defined inputs to organize .yaml file subset into pandas data frame then exports to excel file sheet
    '''

    # Reads in .yaml file
    # Defines initialized objects in the yaml file with objects defined in processing script

    path_yaml = PATH_CONFIG0 / 'about.yaml'
    
    try:
        with open(path_yaml, 'r') as yaml_file:
            yaml_about = yaml.load(yaml_file, Loader=yaml.SafeLoader)
    except FileNotFoundError:
        print(f"Error: The file at {path_yaml} does not exist.")
    except Exception as e:
        print(f"An error occurred: {e}")

    df = pd.DataFrame.from_dict([yaml_about[EST]['ACS']['Income_3']]).T.reset_index().rename(columns = {'index': 'Indicator', 0: 'Income_3'})


    df.loc[df['Indicator'] == 'Last Updated', 'Income_3'] = date.today().strftime('%Y-%m-%d')
    df.loc[df['Indicator'] == 'Year(s)'     , 'Income_3'] = f"{df_inc3['Year'].min()}-{df_inc3['Year'].max()}"


    df.loc[df['Indicator'] == 'Geography', 'Income_3'] = GEO

    if GEO == 'Places':
        df.loc[df['Indicator'] == 'Geography', 'Income_3'] = 'Census Designated Places (Jurisdictions)'


    # Split notes into rows, for visual clarity in about
    # Find the row with 'Notes', then use that to take the information
    # Separate based off of NewLines, make the rows with this
    # Make a blank row past the first one. This way, we don't have to see notes as a cell like 7 times.
    # Create a df from the new separated rows. Drop the old notes row
    # Combine original with new rows
    # Finally, we split the notes
    def split_notes(df):
        row_notes = df[df['Indicator'] == 'Notes'].copy()
        notes = row_notes['Income_3'].values[0]
        
        lines = notes.split('\\n')
        rows_new = [{'Indicator': 'Notes' if i == 0 else '', 'Income_3': line} for i, line in enumerate(lines) if line]
        
        df_new = pd.DataFrame(rows_new)
        df_filtered = df[df['Indicator'] != 'Notes']
       
        df_notes = pd.concat([df_filtered, df_new], ignore_index=True)
        
        return df_notes
    

    df = split_notes(df)

    return df



geographies=['Block Groups', 'Tracts', 'Counties', 'MSA', 'States', 'National',
             'Places', 'Congressional Districts', 'State Legislative Lower Districts', 'State Legislative Upper Districts']

file_inc = PATH_SERVER / f'Income_1 MPO {EST}.xlsx'
df_mpo = pd.read_excel(file_inc, sheet_name='MPO')
df_mpo = df_mpo[df_mpo['Race/Ethnicity'] == 'All'][['Year', 'Median Household Income']].drop_duplicates().reset_index(drop=True)
df_mpo = df_mpo.rename(columns={'Median Household Income':'Regional Median Household Income'})
display(df_mpo)



if GEO in ['Congressional Districts', 'State Legislative Lower Districts', 'State Legislative Upper Districts']:
    if GEO == 'Congressional Districts': sheet_name='CD'
    if GEO == 'State Legislative Lower Districts': sheet_name='SLDL'
    if GEO == 'State Legislative Upper Districts': sheet_name='SLDU'
else:
    sheet_name=GEO

file_in = PATH_SERVER / f'Income_1 {GEO} {EST}.xlsx'
df=pd.read_excel(file_in, sheet_name=sheet_name)

if GEO == 'MSA':
    df = df[df['MSA'].str.contains('Sacramento|Yuba')].reset_index(drop=True)

df = df.merge(df_mpo, on=['Year'], how='left')
df['Percent of Regional Median Household Income'] = (df['Median Household Income']/df['Regional Median Household Income'])
df = clean_fips(df)

display(df)
df_about = write_about(df)
display(df_about)

if EXPORT:

    for path_ in [PATH_SERVER, PATH_SP]:
        file_out = path_/f'Income_3 {GEO} {EST}.xlsx'

        if ABOUT:
            with pd.ExcelWriter(file_out, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
                df_about.to_excel(writer, index=False, sheet_name='About', header=False)
                df.to_excel(writer, index=False, sheet_name=sheet_name)
        else:
            with pd.ExcelWriter(file_out, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
                df.to_excel(writer, index=False, sheet_name=sheet_name)
        
        if ABOUT:            
            if UPDATE:
                file_about = PATH_SP / 'Process Revamp' / 'Task 6. Process Map' / 'About Indicators.xlsx'
                with pd.ExcelWriter(file_about, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
                    df_about.to_excel(writer, index=False, sheet_name='Income_3', header=False)


